# Lesson 16｜資料清理

先找問題、再修正、最後重新驗證。

共 25 個逐步示範；請依序執行。

> 本課已附上 `lesson16_orders_dirty.csv`。只下載單一課程時，請讓 CSV 與 Notebook 保持在同一資料夾。


## 01｜讀取刻意弄髒的資料

先看原始內容，不急著修改。


In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
print(df.head())
print(df.shape)


   Order ID   Order Date Product  Category Quantity  Unit Price City  Payment
0       B001  2026-08-01     美式咖啡       飲品        2        80.0   台北市    CARD
1       B002  2026-08-01      拿鐵        飲品        1       120.0    新北  Mobile
2       B003  2026/08/02       貝果       餐點        3        65.0   台中市    cash
3       B004  2026-08-02       紅茶       飲品        2        55.0   NaN  MOBILE
4       B005  2026-08-03       可頌       餐點        2        70.0    台北     NaN
(24, 8)


## 02｜先保留原始 DataFrame

copy() 讓清理錯誤時可以回頭比較。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
raw_df = df.copy()
clean_df = df.copy()
print(raw_df.shape, clean_df.shape)


## 03｜第一次健康檢查

查看欄名、型態與前幾列。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
print(list(df.columns))
print(df.dtypes)
print(df.head(3))


## 04｜計算每欄缺失值

isna() 找缺值，sum() 計算每欄數量。


In [9]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
print(df.isna().sum())


 Order ID     0
Order Date    0
Product       0
Category      0
Quantity      1
Unit Price    1
City          1
Payment       1
dtype: int64


## 05｜看出哪些列含缺值

axis=1 代表逐列檢查；any() 代表任一欄缺值。


In [3]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
missing_rows = df[df.isna().any(axis=1)]
print(missing_rows)


   Order ID   Order Date Product  Category Quantity  Unit Price City  Payment
3       B004  2026-08-02       紅茶       飲品        2        55.0   NaN  MOBILE
4       B005  2026-08-03       可頌       餐點        2        70.0    台北     NaN
5       B006  2026-08-03       拿鐵       飲品      NaN       120.0   桃園市    card
9       B010  2026-08-05       貝果       餐點        2         NaN   新北市    CARD


## 06｜dropna()：刪除含缺值的列

刪除前後都要確認資料筆數。


In [4]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.dropna()
print("清理前：", len(df))
print("清理後：", len(clean_df))


清理前： 24
清理後： 20


## 07｜只針對重要欄位刪除缺值

subset 決定哪些欄位是必要條件。


In [11]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
#subset的功能是聚焦在後方指定的標籤欄下的資料，若出現NaN才會將該列刪除
clean_df = df.dropna(subset=["Quantity", "Unit Price"] )
print(clean_df.shape)


(22, 8)


## 08｜fillna()：補上文字缺值

付款方式不明時可補成「未知」，但要說明規則。


In [10]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df["Payment"] = clean_df["Payment"].fillna("unknown")
print(clean_df["Payment"].isna().sum())


0


## 09｜用中位數補數字缺值

先轉數字，再用 median() 取得較不受極端值影響的代表值。


In [12]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df["Unit Price"] = pd.to_numeric(clean_df["Unit Price"], errors="coerce")
#to_numeric功能是Pandas 用來將資料轉換為數字（int 或 float）的函數
#當資料中包含無法轉成數字的字串（例如 "100元"、"N/A" 或錯字 "abc"）時：
#若沒有設定：Pandas 會直接報錯並停止執行
#設定為 "coerce"：Pandas 會強制將這些異常文字替換為 NaN（缺失值），不會中斷程式。
#clean_df["Unit Price"] = ...：將轉型完成的結果重新放回 "Unit Price" 欄位中。
median_price = clean_df["Unit Price"].median()
clean_df["Unit Price"] = clean_df["Unit Price"].fillna(median_price)
print(median_price)


80.0


## 10｜找出完整重複列

duplicated() 會標記後面重複出現的列。


In [16]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
print(df.duplicated().sum())
print(df[df.duplicated(keep=False)])


1
    Order ID   Order Date Product  Category Quantity  Unit Price City  Payment
19       B020  2026-08-10      三明治       餐點        2        95.0    台北  mobile
20       B020  2026-08-10      三明治       餐點        2        95.0    台北  mobile


## 11｜刪除完整重複列

drop_duplicates() 預設保留第一筆。


In [17]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.drop_duplicates()
print(len(df), len(clean_df))


24 23


## 12｜依關鍵編號找重複

同一 order_id 即使內容不同，也需要人工確認。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
print(df[df.duplicated(subset=[" Order ID "], keep=False)])


## 13｜清除欄名前後空白

str.strip() 會處理每一個欄名。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.strip()
print(list(clean_df.columns))


## 14｜統一欄名大小寫

lower() 讓欄名全部變小寫。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.strip().str.lower()
print(list(clean_df.columns))


## 15｜把欄名空格改成底線

replace() 讓多字欄名更適合 Python。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
print(list(clean_df.columns))


## 16｜rename()：明確重新命名

用 dict 指定舊欄名與新欄名。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
clean_df = clean_df.rename(columns={"order_date": "date", "unit_price": "price"})
print(list(clean_df.columns))


## 17｜清除文字欄前後空白

Series.str.strip() 只處理字串內容。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df["Product "] = clean_df["Product "].str.strip()
clean_df["City "] = clean_df["City "].str.strip()
print(clean_df[["Product ", "City "]].head())


## 18｜統一城市名稱

把「市」移除，讓台北與台北市成為同一格式。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df["City "] = clean_df["City "].str.strip().str.replace("市", "", regex=False)
print(clean_df["City "].dropna().unique())


## 19｜統一付款方式大小寫

lower() 把 CARD、Card、card 統一。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df["Payment"] = clean_df["Payment"].str.strip().str.lower()
print(clean_df["Payment"].value_counts(dropna=False))


## 20｜to_numeric()：安全轉換數字

errors='coerce' 會把無法轉換的值變成 NaN。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df["Quantity"] = pd.to_numeric(clean_df["Quantity"], errors="coerce")
print(clean_df["Quantity"].isna().sum())


## 21｜找出無法轉換的原始數字

轉換結果是 NaN、原始值卻不是空白，代表格式有問題。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
converted = pd.to_numeric(df["Quantity"], errors="coerce")
invalid_rows = df[converted.isna() & df["Quantity"].notna()]
print(invalid_rows[[" Order ID ", "Quantity"]])


## 22｜處理無效數字

本例選擇移除必要數字無法轉換的列。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df["Quantity"] = pd.to_numeric(clean_df["Quantity"], errors="coerce")
clean_df["Unit Price"] = pd.to_numeric(clean_df["Unit Price"], errors="coerce")
clean_df = clean_df.dropna(subset=["Quantity", "Unit Price"] )
print(clean_df.shape)


## 23｜to_datetime()：轉換日期

先把 `/` 統一成 `-`，再用 errors='coerce' 把真正錯誤的日期轉成 NaT。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
date_text = clean_df["Order Date"].astype("string").str.strip().str.replace("/", "-", regex=False)
clean_df["Order Date"] = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
print(clean_df[["Order Date"]].head())
print("無效日期：", clean_df["Order Date"].isna().sum())


## 24｜找出並處理無效日期

先顯示問題列，再依情境移除。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
date_text = clean_df["Order Date"].astype("string").str.strip().str.replace("/", "-", regex=False)
clean_df["Order Date"] = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
print(clean_df[clean_df["Order Date"].isna()][[" Order ID ", "Order Date"]])
clean_df = clean_df.dropna(subset=["Order Date"])
print(clean_df.shape)


## 25｜完整清理並另存新檔

保留原始 CSV，把乾淨資料寫入新檔。


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("lesson16_orders_dirty.csv")
df = pd.read_csv(data_path)
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
for column in ["product", "city", "payment"]:
    clean_df[column] = clean_df[column].str.strip().str.lower()
clean_df["city"] = clean_df["city"].str.replace("市", "", regex=False)
clean_df["quantity"] = pd.to_numeric(clean_df["quantity"], errors="coerce")
clean_df["unit_price"] = pd.to_numeric(clean_df["unit_price"], errors="coerce")
date_text = clean_df["order_date"].astype("string").str.strip().str.replace("/", "-", regex=False)
clean_df["order_date"] = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
clean_df = clean_df.drop_duplicates().dropna(subset=["quantity", "unit_price", "order_date"])
output_path = Path("lesson16_orders_cleaned.csv")
clean_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("已輸出：", output_path)
print(clean_df.shape)
